# Quantum Volume Characterization

Quantum volume is a benchmark for the effective performance of a quantum processor, capturing the impact of errors across multiqubit circuits. It is based on random circuits and the **heavy output probability**, namely the probability of measuring bitstrings whose ideal probabilities are above the median of the noiseless output distribution. For an ideal device this probability is approximately $0.85$. A width is considered successful if the measured heavy output probability exceeds $2/3$ with a $2\sigma$ confidence margin. The quantum volume is then defined as $\mathrm{QV}=2^n$, where $n$ is the largest width that passes.

***

### The model
For $n$ qubits, the model consists of $n$ layers. In each layer, the qubits are randomly paired, and a Haar-random SU(4) gate is applied to each pair. After all layers, the circuit is measured. The heavy outputs are determined from the noiseless distribution of the same random circuit, which is why the protocol is mainly practical for moderate-size QPUs.

### The protocol

For a fixed width $n$, we generate $T$ independent random model circuits of width $n$ and depth $n$, and execute them on the device. For each circuit, we estimate the heavy-output probability from the measurement data, namely the total probability assigned to outcomes whose ideal probabilities are above the median of the ideal output distribution. We then aggregate this quantity over the $T$ trials and construct a lower confidence bound.

When the number of completed trials is at least $100$, we use the pooled confidence criterion of Cross et al [[1]](#Cross_etal). For smaller runs, we use the simpler normal-approximation lower bound $\bar{h}-2\,\mathrm{SE}(\bar{h})$, where $\bar{h}$ is the mean heavy-output probability across trials and $\mathrm{SE}(\bar{h})$ is its standard error. A width $n$ is considered successful if the resulting lower bound exceeds $2/3$.

The reported quantum volume is $\mathrm{QV}=2^n$, where $n$ is the largest tested width that passes. In particular, if the largest executed width also passes, the true quantum volume is at least this value and may in fact be larger.

***
***

In [1]:
import sys

sys.path.insert(0, "..")

from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

In [2]:
# ============================================================
# Fresh start: reset all generated report/results files
# Uncomment to start a new benchmarking project from scratch
#
# from project_reset import reset_benchmark_project

# reset_benchmark_project()
# ============================================================

### Set a `QuantumVolumeProtocol` with a directory name fixed for the current run

We also need to pass the minimal and maximal number of qubits, and the number of trails (a suggested number is around 100, here we take only 10). 

In [3]:
from pathlib import Path

from protocol import QuantumVolumeProtocol

QV_DESCRIPTION = Path("../descriptions/quantum_volume.tex").read_text(encoding="utf-8")

protocol = QuantumVolumeProtocol(
    min_problem_size=2,
    max_problem_size=4,
    num_trials=10,
    runners=None,  # will be defined later
    results_dir="results/quantum_volume",
    max_submitted_jobs_in_dir=3,
    report_family_title="Quantum Volume",
    report_family_description=QV_DESCRIPTION,
    build_report_each_time=True,
)

In [4]:
# Uncomment to clear the data of a previous run
#
# await protocol.reset_files()

### Choose on which backend to run 

We can print the list of backends:

In [5]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `QuantumVolumeProtocol`.)*

In [6]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [7]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [8]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

In [9]:
protocol.runners = runners

### Run Quantum Volume Protocol

We run the protocol. What happens in practice is that for a given number of qubits, all jobs are sent asynchronously. For convenience we can define a loop with some waiting time (see commented out code). 

In [10]:
import asyncio

from IPython.display import clear_output, display

await protocol.run()

while True:
    await protocol.run()

    summary = protocol.all_width_summaries()

    clear_output(wait=True)

    if (
        not summary.empty
        and (summary["num_completed"] == summary["num_trials_requested"]).all()
    ):
        print("All trials completed.")
        break

    await asyncio.sleep(10)

All trials completed.


Finally, we can print the summary of the results, as well as update the report.

In [11]:
display(protocol.all_width_summaries())
display(protocol.quantum_volume_summary())
display(protocol.final_summary())

,problem_size,backend_service_provider,backend_name,num_trials_requested,num_completed,num_total,mean_score,std_score,stderr_score,lower_confidence_bound,passed
0,2,Classiq,simulator,10,10,10,0.766600,0.091670,0.028989,0.708623,True
1,2,Classiq,simulator_statevector,10,10,10,0.764616,0.091708,0.029001,0.706615,True
2,3,Classiq,simulator,10,10,10,0.827400,0.104057,0.032906,0.761589,True
3,3,Classiq,simulator_statevector,10,10,10,0.825181,0.101196,0.032001,0.761179,True
4,4,Classiq,simulator,10,10,10,0.819200,0.050883,0.016091,0.787019,True
5,4,Classiq,simulator_statevector,10,10,10,0.821193,0.049228,0.015567,0.790058,True


,backend_service_provider,backend_name,largest_passing_width,quantum_volume
0,Classiq,simulator,4,16
1,Classiq,simulator_statevector,4,16


,backend_service_provider,backend_name,largest_passing_width,quantum_volume,total_execution_time
0,Classiq,simulator,4,16,0.381634
1,Classiq,simulator_statevector,4,16,0.380294


## Refrences

<a id='Cross_etal'>[1]</a>: [Andrew W. Cross, Lev S. Bishop, Sarah Sheldon, Paul D. Nation, Jay M. Gambetta. "Validating quantum computers using randomized model circuits." Physical Review A 100, 032328 (2019).](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.100.032328)